# Temporal distribution check — clean threshold=1.4 trigger data
Checks whether the genuinely striking early-August vs September activity
pattern originally found in the contaminated trigger data (Week 14) still
holds using the clean, restored, currently-live 94-event dataset. That
original finding is invalidated and must not be reused without this
fresh check.

In [ ]:
import os
import glob
import pandas as pd
from obspy import UTCDateTime

PROJECT_DIR = os.path.expanduser("~/tvz_project")
TRIGGER_EVENTS_DIR = os.path.join(PROJECT_DIR, "quakemigrate", "runs", "tvz_full", "trigger", "events")

all_times = []
for f in sorted(glob.glob(os.path.join(TRIGGER_EVENTS_DIR, "*_TriggeredEvents.csv"))):
    df = pd.read_csv(f)
    if len(df) == 0:
        continue
    all_times.extend(UTCDateTime(t) for t in df["CoaTime"])

print(f"Total triggered events in the live dataset: {len(all_times)}")
assert len(all_times) == 94, "Expected 94 events - check that the 1.4 threshold data is correctly restored"
print("Confirmed: matches the expected clean 94-event count")

In [ ]:
# Split by calendar month and compare
times_df = pd.DataFrame({"time": [t.datetime for t in all_times]})
times_df["date"] = pd.to_datetime(times_df["time"])
times_df["month"] = times_df["date"].dt.month
times_df["day"] = times_df["date"].dt.date

august = times_df[times_df["month"] == 8]
september = times_df[times_df["month"] == 9]

print(f"August events: {len(august)} ({100*len(august)/len(times_df):.1f}%)")
print(f"September events: {len(september)} ({100*len(september)/len(times_df):.1f}%)")
print(f"\nAugust daily rate: {len(august)/31:.2f} events/day")
print(f"September daily rate: {len(september)/30:.2f} events/day")

print("\nEvents by day (all 94):")
daily_counts = times_df.groupby("day").size()
print(daily_counts.to_string())